# DS201 - Lab 3: RNN cho phân loại văn bản và gán nhãn chuỗi

Thứ tự: cài đặt → kiểm tra dữ liệu → (tùy chọn) nạp checkpoint cũ → kiểm tra 1 batch → train từng bài.

## 1. GPU

In [1]:
import torch
print(torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: chưa bật GPU (Settings > Accelerator > GPU T4)")

2.11.0+cu128
CUDA: True
Tesla T4


## 2. Lấy code từ GitHub

In [2]:
import os, glob

REPO_URL = "https://github.com/tnhyen/DS201-Deep-Learning.git"
REPO_DIR = "/kaggle/working/DS201-Deep-Learning"
LAB_SUBDIR = "BTTH3"  

if "<username>" in REPO_URL:
    raise ValueError("Chưa sửa REPO_URL thành link GitHub thật.")

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

if LAB_SUBDIR:
    LAB_DIR = os.path.join(REPO_DIR, LAB_SUBDIR)
else:
    found = [os.path.dirname(p) for p in glob.glob(f"{REPO_DIR}/**/assignment_1.py", recursive=True)
             if "Sample" not in p]
    if len(found) != 1:
        raise RuntimeError(f"Tìm thấy {len(found)} thư mục chứa assignment_1.py: {found}. Hãy điền LAB_SUBDIR.")
    LAB_DIR = found[0]

if not os.path.exists(os.path.join(LAB_DIR, "assignment_1.py")):
    raise FileNotFoundError(f"Không thấy assignment_1.py trong {LAB_DIR}")

%cd {LAB_DIR}
!ls
!pip install -q seqeval   # cần cho Bài 3

/kaggle/working/DS201-Deep-Learning/BTTH3
assignment_1.py  assignment_3.py  data_utils  README.md
assignment_2.py  check_batch.py   models      run_kaggle.ipynb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


## 3. Dữ liệu
- UIT-VSFC: tự tải từ Google Drive của đề bài bằng `gdown` vào `/kaggle/working/UIT-VSFC/`.
- PhoNER-COVID19: tự clone từ GitHub của VinAI, dùng bản **word**.

Thiếu file nào thì cell sẽ dừng với lỗi.

In [3]:
UIT_DIR = "/kaggle/working/UIT-VSFC"
NER_DIR = "/kaggle/working/PhoNER_COVID19/data/word"

if not os.path.exists(os.path.join(UIT_DIR, "UIT-VSFC-train.json")):
    !pip install -q gdown
    import gdown
    gdown.download_folder(
        url="https://drive.google.com/drive/folders/1rdcXNGt_3-QUvV8EtSvVsLMVeHmk6Yqk",
        output=UIT_DIR, quiet=False, use_cookies=False
    )

if not os.path.exists("/kaggle/working/PhoNER_COVID19"):
    !git clone -q https://github.com/VinAIResearch/PhoNER_COVID19.git /kaggle/working/PhoNER_COVID19

paths = [os.path.join(UIT_DIR, f"UIT-VSFC-{split}.json") for split in ["train", "dev", "test"]]
paths += [os.path.join(NER_DIR, f"{split}_word.json") for split in ["train", "dev", "test"]]

for path in paths:
    print("OK     " if os.path.exists(path) else "MISSING", path)

missing = [path for path in paths if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

Retrieving folder contents


Processing file 1UWbzHFeUPu4AuofOEy0Fo0SFotVZHzUe UIT-VSFC-dev.json
Processing file 1g4imdabNbswIu91K6zKFGalkjDBBqaQq UIT-VSFC-test.json
Processing file 1eJdEStCQBf3GtbOIQfIR2iS-Wb3nlfJX UIT-VSFC-train.json


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1UWbzHFeUPu4AuofOEy0Fo0SFotVZHzUe
To: /kaggle/working/UIT-VSFC/UIT-VSFC-dev.json
100%|██████████| 274k/274k [00:00<00:00, 3.09MB/s]
Downloading...
From: https://drive.google.com/uc?id=1g4imdabNbswIu91K6zKFGalkjDBBqaQq
To: /kaggle/working/UIT-VSFC/UIT-VSFC-test.json
100%|██████████| 559k/559k [00:00<00:00, 2.76MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eJdEStCQBf3GtbOIQfIR2iS-Wb3nlfJX
To: /kaggle/working/UIT-VSFC/UIT-VSFC-train.json
100%|██████████| 2.02M/2.02M [00:00<00:00, 13.6MB/s]
Download completed


OK      /kaggle/working/UIT-VSFC/UIT-VSFC-train.json
OK      /kaggle/working/UIT-VSFC/UIT-VSFC-dev.json
OK      /kaggle/working/UIT-VSFC/UIT-VSFC-test.json
OK      /kaggle/working/PhoNER_COVID19/data/word/train_word.json
OK      /kaggle/working/PhoNER_COVID19/data/word/dev_word.json
OK      /kaggle/working/PhoNER_COVID19/data/word/test_word.json


## 4. (Tùy chọn) Nạp checkpoint từ phiên trước
Upload `checkpoints.zip` (đã tải về) thành Kaggle Dataset, add vào notebook rồi điền `OLD_CKPT_DIR`.

Mỗi bài chỉ được chạy tiếp khi có **đủ cả** `<model>_last.pt` (train tiếp) và `<model>_best.pt` (để test). Muốn train lại từ đầu thì thêm `--no_resume` vào lệnh chạy bài đó.

In [4]:
import shutil

OLD_CKPT_DIR = ""   # ví dụ: "/kaggle/input/ds201-lab3-checkpoints"
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def find_file(root, filename):
    found = glob.glob(f"{root}/**/{filename}", recursive=True)
    return found[0] if found else None

if OLD_CKPT_DIR:
    for name in ["lstm", "gru", "bilstm"]:
        files = {kind: find_file(OLD_CKPT_DIR, f"{name}_{kind}.pt") for kind in ["last", "best"]}
        if files["last"] and files["best"]:
            for src in files.values():
                shutil.copy2(src, CKPT_DIR)
            results = find_file(OLD_CKPT_DIR, f"{name}_results.json")
            if results:
                shutil.copy2(results, CKPT_DIR)
            print(f"{name}: đã nạp last + best -> sẽ chạy tiếp")
        elif files["last"] or files["best"]:
            print(f"{name}: THIẾU file ({files}) -> không nạp, bài này sẽ train từ đầu")
        else:
            print(f"{name}: chưa có checkpoint -> train từ đầu")

!ls -la {CKPT_DIR}

total 8
drwxr-xr-x 2 root root 4096 Nov  3 11:31 .
drwxr-xr-x 6 root root 4096 Nov  3 11:31 ..


## 5. Kiểm tra 1 batch trước khi train
Đọc dữ liệu, kiểm tra shape / nhãn / padding, forward và tính loss cho cả 3 model. Mất vài giây.

In [5]:
!python check_batch.py --uit_dir {UIT_DIR} --ner_dir {NER_DIR}

Device: cuda
==================== UIT-VSFC (Bài 1, 2) ====================
train: 11426 mẫu
dev: 1583 mẫu
test: 3166 mẫu
Vocab size: 2487 | Labels (4): {'facility': 0, 'lecturer': 1, 'others': 2, 'training_program': 3}
input_ids: (32, 39) | lengths: (32,) | label: (32,)
LstmModel: logits (32, 4) | loss = 1.3947 -> OK
GruModel: logits (32, 4) | loss = 1.3492 -> OK
==================== PhoNER-COVID19 (Bài 3) ====================
train: 5027 câu
dev: 2000 câu
test: 3000 câu
Vocab size: 5243 | Số nhãn: 20
Nhãn: ['B-AGE', 'B-DATE', 'B-GENDER', 'B-JOB', 'B-LOCATION', 'B-NAME', 'B-ORGANIZATION', 'B-PATIENT_ID', 'B-SYMPTOM_AND_DISEASE', 'B-TRANSPORTATION', 'I-AGE', 'I-DATE', 'I-JOB', 'I-LOCATION', 'I-NAME', 'I-ORGANIZATION', 'I-PATIENT_ID', 'I-SYMPTOM_AND_DISEASE', 'I-TRANSPORTATION', 'O']
input_ids: (32, 95) | lengths: (32,) | tags: (32, 95)
BiLstmNER: logits (32, 95, 20) | loss = 3.0457 -> OK
Tất cả kiểm tra đều OK, có thể bắt đầu train.


## 6. Bài 1 - LSTM trên UIT-VSFC

In [6]:
!python assignment_1.py --data_dir {UIT_DIR} --checkpoint_dir {CKPT_DIR}

Loading vocab ...
Creating datasets ...
Train: 11426 | Dev: 1583 | Test: 3166
Vocab size: 2487 | Labels: {'facility': 0, 'lecturer': 1, 'others': 2, 'training_program': 3}
Creating dataloaders ...
Initializing model and utilities ...
Epoch 1 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 191.28it/s]
F1: 0.5902
Đã lưu best checkpoint (epoch 1)
Epoch 1: train loss = 0.5415 | dev F1 = 0.5902 | patience = 0
Epoch 2 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 225.64it/s]
F1: 0.6960
Đã lưu best checkpoint (epoch 2)
Epoch 2: train loss = 0.4043 | dev F1 = 0.6960 | patience = 0
Epoch 3 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 199.54it/s]
F1: 0.7428
Đã lưu best checkpoint (epoch 3)
Epoch 3: train loss = 0.3493 | dev F1 = 0.7428 | patience = 0
Epoch 4 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 212.13it/s]
F1: 0.7294
Epoch 4: train loss = 0.3100 | dev F1 = 0.7294 | patience = 1
Epoch 5 - Evaluating: 100%|████████████████████| 50

## 7. Bài 2 - GRU trên UIT-VSFC

In [7]:
!python assignment_2.py --data_dir {UIT_DIR} --checkpoint_dir {CKPT_DIR}

Loading vocab ...
Creating datasets ...
Train: 11426 | Dev: 1583 | Test: 3166
Vocab size: 2487 | Labels: {'facility': 0, 'lecturer': 1, 'others': 2, 'training_program': 3}
Creating dataloaders ...
Initializing model and utilities ...
Epoch 1 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 205.11it/s]
F1: 0.6884
Đã lưu best checkpoint (epoch 1)
Epoch 1: train loss = 0.4857 | dev F1 = 0.6884 | patience = 0
Epoch 2 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 191.37it/s]
F1: 0.7057
Đã lưu best checkpoint (epoch 2)
Epoch 2: train loss = 0.3581 | dev F1 = 0.7057 | patience = 0
Epoch 3 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 201.13it/s]
F1: 0.7044
Epoch 3: train loss = 0.3173 | dev F1 = 0.7044 | patience = 1
Epoch 4 - Evaluating: 100%|████████████████████| 50/50 [00:00<00:00, 187.26it/s]
F1: 0.7153
Đã lưu best checkpoint (epoch 4)
Epoch 4: train loss = 0.2885 | dev F1 = 0.7153 | patience = 0
Epoch 5 - Evaluating: 100%|████████████████████| 50

## 8. Bài 3 - BiLSTM Encoder trên PhoNER-COVID19

In [8]:
!python assignment_3.py --data_dir {NER_DIR} --checkpoint_dir {CKPT_DIR}

Loading vocab ...
Creating datasets ...
Train: 5027 | Dev: 2000 | Test: 3000
Vocab size: 5243 | Số nhãn: 20
Creating dataloaders ...
Initializing model and utilities ...
Epoch 1 - Evaluating: 100%|█████████████████████| 63/63 [00:00<00:00, 76.89it/s]
Entity F1: 0.5168
Đã lưu best checkpoint (epoch 1)
Epoch 1: train loss = 0.8287 | dev F1 = 0.5168 | patience = 0
Epoch 2 - Evaluating: 100%|█████████████████████| 63/63 [00:00<00:00, 92.82it/s]
Entity F1: 0.8001
Đã lưu best checkpoint (epoch 2)
Epoch 2: train loss = 0.1974 | dev F1 = 0.8001 | patience = 0
Epoch 3 - Evaluating: 100%|█████████████████████| 63/63 [00:00<00:00, 90.15it/s]
Entity F1: 0.8436
Đã lưu best checkpoint (epoch 3)
Epoch 3: train loss = 0.0923 | dev F1 = 0.8436 | patience = 0
Epoch 4 - Evaluating: 100%|█████████████████████| 63/63 [00:00<00:00, 91.58it/s]
Entity F1: 0.8656
Đã lưu best checkpoint (epoch 4)
Epoch 4: train loss = 0.0624 | dev F1 = 0.8656 | patience = 0
Epoch 5 - Evaluating: 100%|█████████████████████| 63/6

## 9. Tổng hợp kết quả và đóng gói checkpoint
Tải `/kaggle/working/checkpoints.zip` về ở mục Output, hoặc *Save Version* để giữ lại.

In [9]:
import json

for path in sorted(glob.glob(f"{CKPT_DIR}/*_results.json")):
    print(json.dumps(json.load(open(path)), indent=2, ensure_ascii=False))

!cd /kaggle/working && zip -qr checkpoints.zip checkpoints && ls -lh checkpoints.zip

{
  "model": "Encoder 5 BiLSTM layers, hidden 256",
  "dataset": "PhoNER-COVID19 (word-level)",
  "best_epoch": 10,
  "dev_entity_f1": 0.8851446599986574,
  "test_entity_micro_f1": 0.8737095664143153,
  "test_entity_macro_f1": 0.7957956485200488
}
{
  "model": "GRU 5 layers, hidden 256",
  "dataset": "UIT-VSFC (topic)",
  "best_epoch": 14,
  "dev_macro_f1": 0.764238824448277,
  "test_macro_f1": 0.7407430103901282
}
{
  "model": "LSTM 5 layers, hidden 256",
  "dataset": "UIT-VSFC (topic)",
  "best_epoch": 3,
  "dev_macro_f1": 0.7428467872536728,
  "test_macro_f1": 0.7403670872916661
}
-rw-r--r-- 1 root root 207M Nov  3 11:36 checkpoints.zip
